In [1]:
import asyncio
import subprocess
from itertools import chain
from pathlib import Path

from IPython.display import Markdown, display
from tqdm.notebook import tqdm

import ghapicache
import release_notes

%load_ext autoreload
%autoreload 1
%aimport githelpers
%aimport ghapicache
%aimport release_notes

In [2]:
import githelpers

# Reload token
cached = ghapicache.GhApiCache(token=ghapicache._load_ghapi_token(), repo="g4vg")
githelpers.REPO = Path.home() / "Code/g4vg"
ghapi = cached.api

Failed to load cache: [Errno 2] No such file or directory: 'data/ghapicache-celeritas-project-g4vg.json'


In [3]:
# Load teams
TEAMS = {}
team_metadata = await cached.async_teams()
team_results = await asyncio.gather(
    *(release_notes.async_get_team(cached, t) for t in team_metadata)
)
for t, team in zip(team_metadata, team_results):
    TEAMS[t["name"]] = team

team_members = frozenset(chain(*(t.members for t in TEAMS.values())))
org_members = frozenset(m["login"] for m in await cached.async_org_members())
TEAMS["affiliates"] = release_notes.Team(
    description="Associated with Celeritas but not core members",
    members=(org_members - team_members),
)

# Initialize the user cache
local_repo = Path("/Users/seth/Code/celeritas-temp")
user_cache = release_notes.UserCache(cached, local_repo / "scripts/release/users.json")
await asyncio.gather(*(user_cache.aload(login) for login in org_members))

[UserInfo(name='Rashika Gupta', institute=None, email=None, orcid=None, github='Rashika-Gupta'),
 UserInfo(name='Ilker Parmaksiz', institute=None, email='ip33@rice.edu', orcid=None, github='ilkerparmaksiz'),
 UserInfo(name='Ben Morgan', institute=None, email=None, orcid=None, github='drbenmorgan'),
 UserInfo(name='Elliott Biondo', institute=None, email='elliott.biondo@gmail.com', orcid=None, github='elliottbiondo'),
 UserInfo(name='Hayden Hollenbeck', institute=None, email=None, orcid=None, github='hhollenb'),
 UserInfo(name='Paul Romano', institute=None, email='promano@anl.gov', orcid=None, github='paulromano'),
 UserInfo(name='Amanda Lund', institute=None, email='alund@anl.gov', orcid=None, github='amandalund'),
 UserInfo(name='Loren Schwiebert', institute=None, email='loren@wayne.edu', orcid=None, github='LSchwiebert'),
 UserInfo(name=None, institute=None, email=None, orcid=None, github='aghoshpub'),
 UserInfo(name='Owen Strong', institute=None, email=None, orcid=None, github='osans

# Draft github release

In [4]:
release_md = release_notes.ReleaseMetadata(
    release="1.0.8", merge_bases=["v1.0.7"], target_branch="upstream/main"
)

In [10]:
release_md = release_notes.create_local_tag(release_md)

Updating release target from upstream/main to 8261eec419872ede1ad5f8528a6295d5d21db400


In [6]:
import tempfile

tgz_data = githelpers.git_archive_tgz(release_md.target_branch)
tgz_name = f"{cached.repo}-{release_md.release}.tar.gz"

with tempfile.TemporaryDirectory(delete=False) as tmpdir:
    with open(Path(tmpdir) / tgz_name, "wb") as f:
        f.write(tgz_data)
    subprocess.run(["open", tmpdir])

Decompress and smoke check the resulting gzip file, then:

In [7]:
subprocess.run(
    ["cmake", "-P", "cmake/CgvFindVersion.cmake"],
    cwd=Path(tmpdir) / f"{cached.repo}-{release_md.release}",
)

VERSION="1.0.8"
VERSION_STRING="1.0.8"


CompletedProcess(args=['cmake', '-P', 'cmake/CgvFindVersion.cmake'], returncode=0)

In [11]:
ghapi = cached.api

gh_release = release_notes.find_release(ghapi, release_md.release)
if not gh_release:
    gh_release = release_notes.create_release(ghapi, release_md, "")
    tgz_asset = release_notes.upload_asset(cached, gh_release, tgz_data, tgz_name)

No release found for version: 1.0.8
Draft release 1.0.8 created: https://github.com/celeritas-project/g4vg/releases/tag/untagged-ac354085dfc70024c43c
Uploaded artifact: https://github.com/celeritas-project/g4vg/releases/download/untagged-ac354085dfc70024c43c/g4vg-1.0.8.tar.gz


In [12]:
sha256_hash = release_notes.prepare_spack_update("g4vg", release_md, tgz_data)

Spack version copied to clipboard!


In [20]:
display(
    Markdown(f"""
## Time to publish!

1. Visit [the github release page]({gh_release["html_url"]})
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is {sha256_hash}
4. Double check the name and release notes
5. Edit and publish!
""")
)


## Time to publish!

1. Visit [the github release page](https://github.com/celeritas-project/g4vg/releases/tag/untagged-6201159970f90ae9c66e)
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is fb70f509cc85b0d5311b70246e141ffdd47abf9b175cf95aa0fd1b75edae616e
4. Double check the name and release notes
5. Edit and publish!


# Push to zenodo

In [13]:
import zenodoapi

%aimport zenodoapi

In [14]:
# Load the Zenodo token
token_path = Path.home() / ".config/zenodo-token"
with open(token_path) as f:
    zenodo_token = f.read().strip()

zenodo = zenodoapi.Zenodo(zenodo_token)
zenodo.api_url = "https://zenodo.org/api/"

In [15]:
PR_PARSER = release_notes.PullRequestLogParser(None)


async def load_contributions(release_md):
    prs = release_notes.PullRequestRange(release_md, PR_PARSER)
    contributions = await release_notes.async_load_contributions_batch(
        cached, prs.pull_ids
    )
    return release_notes.merge_contributions(contributions)


def download_tarball(gh_release):
    asset = gh_release["assets"][0]
    name = asset["name"]
    content = cached.download_file(asset["url"], ext=Path(name).suffix)
    assert content is not None
    return (content, name)


make_zenodo_md = release_notes.ZenodoMetadataBuilder(user_cache=user_cache, teams=TEAMS)

## Minor release

This assumes the major release has already been published upstream into Zenodo.

In [17]:
assert not release_md.is_major()
major, minor, patch = release_md.as_version()
major_dep = zenodo.find_deposition(f"G4VG {major}.{minor}")

In [18]:
release_md

ReleaseMetadata(release='1.0.8', merge_bases=['v1.0.7'], target_branch='8261eec419872ede1ad5f8528a6295d5d21db400')

In [19]:
zenodo.get_deposition(major_dep.get_latest_version().id)

ZenodoDeposition(id=18273294, title="G4VG 1.0")

In [20]:
gh_release = release_notes.find_release(ghapi, release_md.release)

In [21]:
# Get the Zenodo metadata
from requests import HTTPError

old_md = major_dep.data["metadata"]
new_md = make_zenodo_md(await load_contributions(release_md), release_md, gh_release)
# Only update editors, not team members
old_contrib = new_md["contributors"]
new_contrib = [u for u in old_contrib if u["type"] == "Editor"]
old_contrib = [u for u in old_contrib if u["type"] != "Editor"]
new_md["contributors"] = new_contrib + old_contrib
# Don't change the title
new_md["title"] = major_dep.data["metadata"]["title"]
# Add the body
new_md["description"] = "\n\n".join([
    old_md["description"],
    f"<h1>Version {release_md.release}</h1>",
    new_md["description"],
])
try:
    new_vers = major_dep.create_new_version()
except HTTPError as e:
    if e.response.status_code == 400:
        print("Draft may already exist:", e)
        new_vers = major_dep.get_latest_draft()
    else:
        raise
try:
    new_vers.update(new_md)
except HTTPError as e:
    print("Draft may already exist:", e)

(content, name) = download_tarball(gh_release)
new_vers.upload(content, name)
new_vers.refresh()
for file in new_vers.get_files():
    if file.filename != name:
        try:
            file.delete()
        except Exception as e:
            print(f"Failed to delete {file.filename}: {e}")
new_vers.refresh()

ValueError: Missing properties from user mdemarteau: UserInfo(name=None, institute=None, email=None, orcid=None, github='mdemarteau')

## Major release

In [57]:
assert release_md.is_major()
major, minor, patch = release_md.as_version()
release_md = release_notes.ReleaseMetadata.from_comprehensive_version(minor)
title = f"Celeritas {major}.{minor}"

In [ ]:
zmd = make_zenodo_md(await load_contributions(release_md), release_md, gh_release)
zmd["title"] = title

# Create the Zenodo deposition
deposition = zenodo.create_deposition(zmd)

  0%|          | 0/234 [00:00<?, ?it/s]

Created deposition 15281110 at https://zenodo.org/deposit/15281110 : Celeritas 0.6


In [60]:
# Upload the release tarball
deposition.upload(artifact_tgz, f"celeritas-{release_md.release}.tar.gz")

print(deposition.html)
print("""
IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses
""")

Uploaded celeritas-0.6.0.tar.gz: version bbd63016-30a1-4be9-9a74-329ca975bed9
https://zenodo.org/deposit/15281110

IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses

